# Exploring the Relationship between Health and Educational Outcomes
**Data Analysis with Python — Group Project**  
Cosimo ZATTI, William Manca di Villahermosa, Robert BIRKMAYER

This notebook investigates the association between three OECD health indicators (infant mortality, life expectancy at birth, health expenditure as % of GDP) and country-level PISA 2022 scores in mathematics, reading, and science.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- File paths (relative to the notebook — works after cloning the repo) ---
PISA_FILE = Path('xluqor.xlsx')

LIFE_EXP_FILE    = Path('OECD.ELS.HD,DSD_HEALTH_STAT@DF_LE,1.1,filtered,2026-04-23 15-20-54.xlsx')
INFANT_MORT_FILE = Path('OECD.ELS.HD,DSD_HEALTH_STAT@DF_MIM,1.1,filtered,2026-04-23 15-22-00.xlsx')
HEALTH_EXP_FILE  = Path('OECD.ELS.HD,DSD_SHA@DF_SHA,1.0,filtered,2026-04-23 15-25-18.xlsx')

## 2. Load PISA 2022 Data

Source: OECD PISA 2022 Results (Volume I), Tables I.2.1–I.2.3.  
Each table ranks countries by mean score in one subject. We extract country name and mean score for math, reading, and science.

In [ ]:
def load_pisa_subject(filepath, sheet_name, score_col):
    """
    Load one PISA subject table.
    
    The Excel layout has metadata rows 0-9, then a header row at row 10,
    then country data from row 11 onward. Columns:
      col 0 = mean score, col 1 = country name, col 2 = comparison list (unused).
    """
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=10, engine='calamine')
    # Row 0 is still a text-header artefact from the Excel layout — drop it
    df = raw.iloc[1:, :2].copy()
    df.columns = [score_col, 'country']
    # Strip asterisks used in PISA to mark annex/footnote countries
    df['country'] = df['country'].astype(str).str.replace('*', '', regex=False).str.strip()
    df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
    return df[['country', score_col]].dropna()


math_df    = load_pisa_subject(PISA_FILE, 'Table I.2.1', 'math_score')
reading_df = load_pisa_subject(PISA_FILE, 'Table I.2.2', 'reading_score')
science_df = load_pisa_subject(PISA_FILE, 'Table I.2.3', 'science_score')

# Merge the three subjects into one PISA dataframe
pisa = (
    math_df
    .merge(reading_df, on='country')
    .merge(science_df, on='country')
)

print(f"PISA 2022: {len(pisa)} countries/economies")
pisa.head(10)

## 3. Load OECD Health Statistics (2022)

Three indicators from the OECD Health Statistics database, filtered to the 2022 observation year:
- **Life expectancy at birth** (years)
- **Infant mortality rate** (deaths per 1,000 live births)
- **Health expenditure** (% of GDP)

In [ ]:
def load_oecd_health(filepath, col_name, year='2022'):
    """
    Load a single OECD health indicator for a given year.

    The OECD export format has 5 metadata rows before the data table.
    Country names are in the 'Time period' column. Section-header rows
    (e.g. 'Sex: Total', 'Measure: Life expectancy') contain a colon and
    are excluded. Some files repeat data for multiple breakdowns (sex,
    gestation threshold); we keep only the first occurrence per country,
    which corresponds to the default Total / all-threshold category.
    """
    raw = pd.read_excel(filepath, header=5, engine='calamine')
    df = raw[['Time period', year]].copy()
    df.columns = ['country', col_name]

    # Drop section-header and divider rows
    skip_exact = {'Reference area', 'Non-OECD economies', '© Terms & conditions'}
    mask_colon = df['country'].astype(str).str.contains(':', na=False)
    mask_skip  = df['country'].astype(str).str.strip().isin(skip_exact)
    df = df[~mask_colon & ~mask_skip]

    # Strip bullet '·' prefix used by OECD for non-member economies
    df['country'] = df['country'].astype(str).str.replace('·', '', regex=False).str.strip()
    df[col_name]  = pd.to_numeric(df[col_name], errors='coerce')
    df = df.dropna()

    # One row per country — keep first (= Total / default breakdown)
    df = df.drop_duplicates(subset='country', keep='first')
    return df.reset_index(drop=True)


life_exp    = load_oecd_health(LIFE_EXP_FILE,    'life_expectancy')
infant_mort = load_oecd_health(INFANT_MORT_FILE, 'infant_mortality')
health_exp  = load_oecd_health(HEALTH_EXP_FILE,  'health_exp_pct_gdp')

# Merge health indicators (outer join preserves all countries)
health = (
    life_exp
    .merge(infant_mort, on='country', how='outer')
    .merge(health_exp,  on='country', how='outer')
)

print(f"OECD health data: {len(health)} countries")
health.head(10)

## 4. Merge and Clean the Dataset

Both datasets use OECD country names, so we merge directly on the `country` column.  
We keep only observations with complete data on all variables (inner join + dropna).

In [ ]:
df = pisa.merge(health, on='country', how='inner').dropna()
df = df.reset_index(drop=True)

print(f"Final dataset: {len(df)} countries with complete data on all variables")
print(f"Variables: {list(df.columns)}")
df

## 5. Descriptive Statistics

In [ ]:
df.describe().round(2)

In [ ]:
# Correlation matrix
corr_cols = ['math_score', 'reading_score', 'science_score',
             'life_expectancy', 'infant_mortality', 'health_exp_pct_gdp']

corr = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, ax=ax, fmt='.2f',
            linewidths=0.5, square=True)
ax.set_title('Correlation matrix — PISA scores and health indicators (2022)', pad=12)
plt.tight_layout()
plt.show()

## 6. Visualisation: Facet Grid

Each panel shows the scatter plot of one health indicator (rows) against one PISA subject score (columns), with an OLS regression line and 95% confidence interval.

In [ ]:
# Reshape to long format for a clean FacetGrid
health_indicators = {
    'life_expectancy':    'Life Expectancy at Birth (years)',
    'infant_mortality':   'Infant Mortality (per 1,000 live births)',
    'health_exp_pct_gdp': 'Health Expenditure (% of GDP)',
}

pisa_subjects = {
    'math_score':    'Mathematics',
    'reading_score': 'Reading',
    'science_score': 'Science',
}

fig, axes = plt.subplots(
    nrows=len(health_indicators),
    ncols=len(pisa_subjects),
    figsize=(14, 12),
    sharex='col',
)

for row_idx, (health_var, health_label) in enumerate(health_indicators.items()):
    for col_idx, (pisa_var, pisa_label) in enumerate(pisa_subjects.items()):
        ax = axes[row_idx, col_idx]

        sns.regplot(
            data=df,
            x=pisa_var,
            y=health_var,
            ax=ax,
            scatter_kws=dict(s=40, alpha=0.75, edgecolors='white', linewidths=0.5),
            line_kws=dict(color='firebrick', linewidth=1.5),
            ci=95,
        )

        # Annotate a few notable countries
        highlight = ['United States', 'Japan', 'Korea', 'Finland', 'Mexico']
        for _, row in df[df['country'].isin(highlight)].iterrows():
            ax.annotate(
                row['country'],
                xy=(row[pisa_var], row[health_var]),
                xytext=(4, 2),
                textcoords='offset points',
                fontsize=7,
                color='#333333',
            )

        # Labels
        if row_idx == len(health_indicators) - 1:
            ax.set_xlabel(pisa_label, fontsize=10)
        else:
            ax.set_xlabel('')

        if col_idx == 0:
            ax.set_ylabel(health_label, fontsize=9)
        else:
            ax.set_ylabel('')

        if row_idx == 0:
            ax.set_title(pisa_label, fontsize=11, fontweight='bold')

fig.suptitle(
    'Health Indicators vs PISA 2022 Scores by Country (OECD)',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('facet_grid.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved to facet_grid.png')

## 7. Correlation Coefficients Summary Table

Pearson correlation coefficients between each health indicator and each PISA subject score.

In [ ]:
summary_rows = []
for health_var, health_label in health_indicators.items():
    for pisa_var, pisa_label in pisa_subjects.items():
        r = df[[health_var, pisa_var]].corr().iloc[0, 1]
        summary_rows.append({
            'Health indicator': health_label,
            'PISA subject':     pisa_label,
            'Pearson r':        round(r, 3),
        })

summary = pd.DataFrame(summary_rows).pivot(
    index='Health indicator', columns='PISA subject', values='Pearson r'
)
summary